# Deep Koopman MPC Portfolio Optimizer
**A Fluid-Dynamics-Inspired Quantitative Trading System for the BSE SENSEX-30**

---

## Theoretical Foundation

Financial markets are chaotic, high-dimensional, non-linear dynamical systems. Standard RL struggles because it must explore a vast action space to discover good policies. This notebook takes a radically different approach inspired by fluid dynamics.

### Core Idea: Koopman Operator Theory
Koopman Theory proves that **any finite-dimensional non-linear system can be lifted into a higher-dimensional linear space** where classical, rigorous optimal control theory applies.

We learn four objects from data:
- **Encoder ϕ**: Maps observable market state **xₜ** → latent fluid state **zₜ** (non-linear neural network)
- **Transition Matrix K**: The "natural current" of the market. How **zₜ** evolves autonomously.
- **Control Matrix B**: How your portfolio weights **uₜ** physically alter the latent state.
- **Return Matrix C**: Linear readout mapping **zₜ** → expected per-stock returns **μₜ**.

The **physics engine** inside the network's bottleneck:
$$\mathbf{z}_{t+1} = \mathbf{K}\mathbf{z}_t + \mathbf{B}\mathbf{u}_t$$

### Three Phases
1. **Phase 1 — System Identification**: Train the Deep Koopman Autoencoder to extract K, B, C from data.
2. **Phase 2 — Model Predictive Control (MPC)**: Use K, B, C to solve a **convex** portfolio optimization problem over a 7-day rolling horizon:
$$J = \max_{\mathbf{u}} \sum_{k=0}^{H} \underbrace{\mathbf{u}_{t+k}^\top (\mathbf{C}\mathbf{z}_{t+k})}_{\text{Expected Return}} - \underbrace{\gamma\, \mathbf{u}_{t+k}^\top \boldsymbol{\Sigma}\, \mathbf{u}_{t+k}}_{\text{Markowitz Risk}} - \underbrace{\lambda \|\mathbf{u}_{t+k} - \mathbf{u}_{t+k-1}\|_1}_{\text{Transaction Costs}}$$
3. **Phase 3 — Convex Execution**: Because **z** propagates linearly and **J** is concave in **u**, CVXPY finds the **guaranteed global optimum** in milliseconds per day.

---
**Universe**: BSE SENSEX-30 | **Period**: 2014–2024 | **Benchmark**: BSE Sensex (^BSESN)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install yfinance cvxpy torch --quiet

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from copy import deepcopy
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

import cvxpy as cp
import yfinance as yf

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {device}')
print(f'PyTorch     : {torch.__version__}')
print(f'CVXPY       : {cp.__version__}')

## Configuration
All hyperparameters are centralised here. Change only this cell to experiment.

In [ ]:
# ── Universe ──────────────────────────────────────────────────────────────────
SENSEX_30 = [
    'RELIANCE.BO',  'TCS.BO',       'HDFCBANK.BO',  'INFY.BO',      'ICICIBANK.BO',
    'HINDUNILVR.BO','ITC.BO',       'SBIN.BO',      'BHARTIARTL.BO','KOTAKBANK.BO',
    'LT.BO',        'AXISBANK.BO',  'ASIANPAINT.BO','MARUTI.BO',    'SUNPHARMA.BO',
    'TITAN.BO',     'BAJFINANCE.BO','WIPRO.BO',     'ONGC.BO',      'NTPC.BO',
    'POWERGRID.BO', 'ULTRACEMCO.BO','NESTLEIND.BO', 'BAJAJFINSV.BO','TATAMOTORS.NS',
    'HCLTECH.BO',   'TATASTEEL.BO', 'JSWSTEEL.BO',  'M&M.BO',       'ADANIENT.BO',
]
SENSEX_BENCHMARK = '^BSESN'

NIFTY_50 = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','SUNPHARMA.NS',
    'TITAN.NS','BAJFINANCE.NS','WIPRO.NS','ONGC.NS','NTPC.NS',
    'POWERGRID.NS','ULTRACEMCO.NS','NESTLEIND.NS','BAJAJFINSV.NS','TATAMOTORS.NS',
    'HCLTECH.NS','TATASTEEL.NS','JSWSTEEL.NS','M&M.NS','ADANIENT.NS',

    'ADANIPORTS.NS','APOLLOHOSP.NS','BAJAJ-AUTO.NS','BPCL.NS','BRITANNIA.NS',
    'CIPLA.NS','COALINDIA.NS','DIVISLAB.NS','DRREDDY.NS','EICHERMOT.NS',
    'GRASIM.NS','HDFCLIFE.NS','HEROMOTOCO.NS','HINDALCO.NS','INDUSINDBK.NS',
    'JSWSTEEL.NS','LTIM.NS','SBILIFE.NS','SHRIRAMFIN.NS','TECHM.NS',
    'UPL.NS'
]
NIFTY_BENCHMARK = '^NSEI'

# ── Date ranges ───────────────────────────────────────────────────────────────
START_DATE  = '2014-01-01'
END_DATE    = '2024-12-31'
TRAIN_END   = '2021-12-31'
VAL_END     = '2022-12-31'
# Test = 2023-01-01 → 2024-12-31

# ── Feature / model geometry ──────────────────────────────────────────────────
WINDOW          = 10    # W: trading-day lookback window per sample
N_FEATURES      = 7     # F: per-stock features (see engineering cell)
STOCK_EMBED_DIM = 64    # per-stock latent dimension (Stage-1 encoder output)
LATENT_DIM      = 64    # portfolio-level latent dim |z_t| (must equal STOCK_EMBED_DIM
                        #  if using simple mean-pool; increase for richer aggregation)
DROPOUT         = 0.10

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 32
N_EPOCHS   = 80
LR         = 3e-4
PATIENCE   = 15

# Loss weights (λ_recon·L_recon + λ_latent·L_latent + λ_fwd·L_fwd + λ_ret·L_ret)
LAMBDA_RECON  = 1.0   # autoencoder reconstruction
LAMBDA_LATENT = 4.0   # Koopman linearisation  ← most important, keep highest
LAMBDA_FWD    = 1.5   # forward prediction in obs. space
LAMBDA_RET    = 2.0   # C-matrix return supervision
LAMBDA_EIGVAL = 0.5   # K eigenvalue stability penalty

# ── MPC (Phase 2) ─────────────────────────────────────────────────────────────
H          = 7       # planning horizon (days)
GAMMA      = 1.0     # Markowitz risk-aversion coefficient
LAMBDA_TC  = 0.001   # transaction cost coefficient (≈ 10 bps per unit turnover)
MAX_WEIGHT = 0.20    # max single-stock weight (long-only mode)
COV_WINDOW = 60      # rolling window (days) for Σ estimation during backtest

# ── Log Return feature index ──────────────────────────────────────────────────
LOGRET_IDX = 0   # feature 0 is log_return (see engineering cell)

print('Configuration loaded.')
print(f'  Universe    : {len(SENSEX_30)} tickers')
print(f'  Latent dim  : {LATENT_DIM}')
print(f'  MPC horizon : H = {H} days')

## Phase 1 — Step 1: Data Pipeline
Download 10 years of OHLCV data, enforce data-quality thresholds, forward-fill small gaps.

In [ ]:
def download_data(tickers, start, end, missing_thresh=0.05):
    """Download OHLCV; drop tickers with > missing_thresh fraction of NaN closes."""
    print(f'Downloading {len(tickers)} tickers  [{start} → {end}] ...')
    raw  = yf.download(tickers, start=start, end=end,
                       auto_adjust=True, progress=True, threads=False)
    ohlcv = raw[['Open', 'High', 'Low', 'Close', 'Volume']]

    close       = ohlcv['Close']
    missing_pct = close.isna().mean()
    bad         = missing_pct[missing_pct > missing_thresh].index.tolist()
    if bad:
        print(f'  Dropping (>{missing_thresh*100:.0f}% missing): {bad}')
        ohlcv = ohlcv.drop(columns=bad, level=1)

    ohlcv = ohlcv.ffill().bfill()
    n_stocks = ohlcv['Close'].shape[1]
    print(f'  Final shape : {ohlcv.shape}  |  stocks retained : {n_stocks}')
    return ohlcv


In [ ]:

ohlcv_data    = download_data(SENSEX_30, START_DATE, END_DATE)
benchmark_raw = yf.download(NIFTY_BENCHMARK, start=START_DATE, end=END_DATE,
                            auto_adjust=True, progress=False)
TICKERS = ohlcv_data['Close'].columns.tolist()
N_STOCKS = len(TICKERS)
print(f'\nN_STOCKS = {N_STOCKS}')
print(f'Tickers  : {TICKERS}')

## Phase 1 — Step 2: Feature Engineering (State Vector **xₜ**)

For each stock, we construct a 7-dimensional daily feature vector:

| # | Feature | Description |
|---|---------|-------------|
| 0 | `log_return` | Daily log return — stationary price proxy |
| 1 | `vol_20` | 20-day rolling std of log returns — realised volatility |
| 2 | `momentum_10` | 10-day cumulative log return — medium-term trend |
| 3 | `rsi_14` | 14-day RSI normalised to [−0.5, +0.5] — mean-reversion signal |
| 4 | `macd_signal` | MACD histogram / price — trend strength |
| 5 | `hl_range` | (High−Low)/Close — intra-day volatility proxy |
| 6 | `volume_z` | 20-day z-score of log volume — liquidity signal |

**Design rationale**: The encoder ϕ ingests **xₜ** and produces **zₜ**. Portfolio weights **uₜ** never touch these raw features — they only enter the latent dynamics via **B**, preventing feature contamination.

In [ ]:
FEATURE_NAMES = ['log_return', 'vol_20', 'momentum_10',
                 'rsi_14', 'macd_signal', 'hl_range', 'volume_z']

In [ ]:

def _rsi(series, period=14):
    d     = series.diff()
    gain  = d.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-d.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / (loss + 1e-9)
    return rs / (1 + rs) - 0.5   # normalise to [-0.5, 0.5]


def compute_features(ohlcv):
    """Returns a 3-D numpy array of shape (T, N_stocks, F)."""
    tickers = ohlcv['Close'].columns.tolist()
    frames  = []

    for tk in tickers:
        c = ohlcv['Close'][tk]
        h = ohlcv['High'][tk]
        l = ohlcv['Low'][tk]
        v = ohlcv['Volume'][tk]

        log_ret  = np.log(c / c.shift(1))
        vol_20   = log_ret.rolling(20).std()
        mom_10   = log_ret.rolling(10).sum()
        rsi      = _rsi(c, 14)
        ema12    = c.ewm(span=12, adjust=False).mean()
        ema26    = c.ewm(span=26, adjust=False).mean()
        macd_h   = (ema12 - ema26) / (c + 1e-9)
        hl_range = (h - l) / (c + 1e-9)
        log_vol  = np.log(v + 1)
        vol_z    = (log_vol - log_vol.rolling(20).mean()) / (log_vol.rolling(20).std() + 1e-9)

        frames.append(pd.DataFrame({
            'log_return'  : log_ret,
            'vol_20'      : vol_20,
            'momentum_10' : mom_10,
            'rsi_14'      : rsi,
            'macd_signal' : macd_h,
            'hl_range'    : hl_range,
            'volume_z'    : vol_z,
        }))

    common_idx = frames[0].index
    for f in frames[1:]:
        common_idx = common_idx.intersection(f.index)

    aligned  = [f.loc[common_idx] for f in frames]
    data_3d  = np.stack([f.values for f in aligned], axis=1)  # (T, N, F)

    first_valid = np.where(~np.isnan(data_3d).any(axis=(1, 2)))[0][0]
    return data_3d[first_valid:], common_idx[first_valid:]


In [ ]:

print('Engineering features...')
data_3d, date_index = compute_features(ohlcv_data)
print(f'3-D tensor (T, N_stocks, F) : {data_3d.shape}')
print(f'Date range                  : {date_index[0].date()} → {date_index[-1].date()}')

## Phase 1 — Step 3: Split, Normalise, Dataset

- **Normalisation**: Computed on train only, applied to all splits (prevents leakage).
- **Dataset**: Yields `(x_t, x_t1, u_t, r_t1)` tuples:
  - `x_t`  — current window `(N, W, F)` (input to encoder)
  - `x_t1` — next window `(N, W, F)` (target for reconstruction + Koopman constraint)
  - `u_t`  — control input `(N,)` used to train the **B** matrix (equal-weight during training)
  - `r_t1` — actual next-day log returns `(N,)` used to supervise the **C** matrix

In [ ]:
# ── Train / Val / Test split ──────────────────────────────────────────────────
train_mask = date_index <= TRAIN_END
val_mask   = (date_index > TRAIN_END) & (date_index <= VAL_END)
test_mask  = date_index > VAL_END

data_train = data_3d[train_mask]
data_val   = data_3d[val_mask]
data_test  = data_3d[test_mask]

for name, d, m in [('Train', data_train, train_mask),
                   ('Val',   data_val,   val_mask),
                   ('Test',  data_test,  test_mask)]:
    print(f'{name:5s}: {d.shape[0]} days  '
          f'({date_index[m][0].date()} → {date_index[m][-1].date()})')

# ── Normalise (train stats only) ──────────────────────────────────────────────
train_mean = np.nanmean(data_train, axis=(0, 1))    # (F,)
train_std  = np.nanstd(data_train,  axis=(0, 1))    # (F,)
train_std  = np.where(train_std < 1e-8, 1e-8, train_std)

def normalise(x): return (x - train_mean) / train_std

data_train_n = normalise(data_train)
data_val_n   = normalise(data_val)
data_test_n  = normalise(data_test)

# ── Save artefacts for later phases ──────────────────────────────────────────
np.save('train_mean.npy',        train_mean)
np.save('train_std.npy',         train_std)
np.save('data_test_n.npy',       data_test_n)
np.save('date_index_test.npy',   date_index[test_mask].astype(str))
np.save('tickers_retained.npy',  np.array(TICKERS))

print(f'\nTrain mean ≈ 0: {data_train_n.mean(axis=(0,1)).round(3)}')
print(f'Train std  ≈ 1: {data_train_n.std( axis=(0,1)).round(3)}')

In [ ]:
class KoopmanDataset(Dataset):
    """
    Sliding-window dataset that yields the four quantities needed
    to train all components of the Koopman system (ϕ, K, B, C, ψ).

    For a window ending at time t (exclusive), the sample is:
      x_t  : (N, W, F)  — observable state at t   → train encoder ϕ
      x_t1 : (N, W, F)  — observable state at t+1 → train K, ψ (Koopman target)
      u_t  : (N,)        — control input at t       → train B
      r_t1 : (N,)        — log returns at t+1        → train C (return readout)

    Control u_t during training:
      We use an equal-weight portfolio (1/N for each stock).  The B matrix
      learns the average market-wide effect of this baseline control.
      During MPC inference, u_t is the decision variable we optimise.
    """
    def __init__(self, data: np.ndarray, window: int):
        assert data.ndim == 3, 'Expected (T, N, F)'
        self.data = torch.tensor(data, dtype=torch.float32)   # (T, N, F)
        self.W    = window
        self.T, self.N, self.F = data.shape
        # valid pairs: x_t ends at i+W-1, x_t1 ends at i+W → i+W+1 ≤ T
        self.n_samples = self.T - self.W - 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Sliding windows — shape (W, N, F) → permute → (N, W, F)
        x_t  = self.data[idx     : idx + self.W    ].permute(1, 0, 2)  # (N, W, F)
        x_t1 = self.data[idx + 1 : idx + self.W + 1].permute(1, 0, 2) # (N, W, F)

        # Equal-weight portfolio as training control signal for B
        u_t = torch.full((self.N,), 1.0 / self.N)

        # Actual log returns for the day following the window (supervise C)
        # day idx + W is the first day *after* the window x_t ends
        r_t1 = self.data[idx + self.W, :, LOGRET_IDX]   # (N,)

        return x_t, x_t1, u_t, r_t1


In [ ]:

train_ds = KoopmanDataset(data_train_n, WINDOW)
val_ds   = KoopmanDataset(data_val_n,   WINDOW)
test_ds  = KoopmanDataset(data_test_n,  WINDOW)

kw = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=(device.type == 'cuda'))
train_loader = DataLoader(train_ds, shuffle=True,  **kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **kw)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')
print(f'Test  samples : {len(test_ds)}')

x_t, x_t1, u_t, r_t1 = train_ds[0]
print(f'\nSample shapes:')
print(f'  x_t  : {tuple(x_t.shape)}   (N_stocks, W, F)')
print(f'  x_t1 : {tuple(x_t1.shape)}')
print(f'  u_t  : {tuple(u_t.shape)}   (N_stocks,)  — control input')
print(f'  r_t1 : {tuple(r_t1.shape)}  (N_stocks,)  — return target for C')
assert not torch.isnan(x_t).any(), 'NaN in x_t'
print('\nNo NaN values detected. ✓')

## Phase 1 — Step 4: Deep Koopman Architecture

### Architecture Overview

```
   xₜ (N, W, F)                  xₜ₊₁ (N, W, F)
       │                               ↑
  ┌────▼──────────────────────────────┼────┐
  │  Stage-1: Per-stock MLP (shared)  │    │
  │  (W·F,) → stock_embed (64,)       │    │
  │  ↓  mean-pool over N stocks       │    │
  │  Stage-2: Portfolio MLP           │    │
  │  (64,) → zₜ (latent_dim,)         │    │
  │           │        │              │    │
  │     K·zₜ + B·uₜ   C·zₜ = μₜ      │    │
  │           │        ↓              │    │
  │           ↓    Expected Returns   │    │
  │      zₜ₊₁_hat                     │    │
  │           │                       │    │
  │  Decoder: (latent_dim,) → (N,W,F) │    │
  └───────────────────────────────────┘    │
                                           │
               ψ(zₜ₊₁_hat) ───────────────┘
```

### The Three Linear Matrices
- **K** `(latent_dim × latent_dim)`: Autonomous market dynamics. Initialised close to identity — "tomorrow ≈ today" warm start. Eigenvalue regularisation keeps spectrum inside the unit circle.
- **B** `(latent_dim × N_stocks)`: Control gain. Maps portfolio weight vector **uₜ** → latent-space perturbation. Trained jointly with the system so the model learns how position-taking shifts the market state.
- **C** `(N_stocks × latent_dim)`: Return readout. **Supervised** by actual next-day log returns, so C learns to extract the return-relevant directions from **z**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENCODER  —  xₜ: (N, W, F)  →  zₜ: (latent_dim,)
# Two-stage: per-stock embedding → portfolio-level projection.
# Using mean-pooling over stocks keeps the encoder permutation-invariant
# (order of stocks doesn't matter) and scales well to different universe sizes.
# ─────────────────────────────────────────────────────────────────────────────
class KoopmanEncoder(nn.Module):
    def __init__(self, window, n_features, stock_embed_dim, latent_dim, dropout=0.1):
        super().__init__()
        in_dim = window * n_features   # 10 × 7 = 70

        # Stage 1: shared per-stock MLP  (W·F) → stock_embed_dim
        self.stock_enc = nn.Sequential(
            nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, stock_embed_dim), nn.LayerNorm(stock_embed_dim),
        )

        # Stage 2: portfolio projection  stock_embed_dim → latent_dim
        self.portfolio_proj = nn.Sequential(
            nn.GELU(),
            nn.Linear(stock_embed_dim, latent_dim),
            # No final activation: latent space must be unconstrained for K to work freely
        )

    def forward(self, x):
        # x: (B, N, W, F)
        B, N, W, F = x.shape
        x_flat = x.reshape(B * N, W * F)             # (B·N, W·F)
        s      = self.stock_enc(x_flat)               # (B·N, stock_embed_dim)
        s      = s.reshape(B, N, -1)                  # (B, N, stock_embed_dim)
        z      = self.portfolio_proj(s.mean(dim=1))   # mean-pool over N → (B, latent_dim)
        return z


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# DECODER  —  zₜ: (latent_dim,)  →  x̂ₜ: (N, W, F)
# Expands from latent state back to the full observable space.
# Shared per-stock decoder mirrors the encoder for gradient symmetry.
# ─────────────────────────────────────────────────────────────────────────────
class KoopmanDecoder(nn.Module):
    def __init__(self, window, n_features, n_stocks, latent_dim, dropout=0.1):
        super().__init__()
        out_dim = window * n_features  # 10 × 7 = 70
        self.W, self.F, self.N = window, n_features, n_stocks

        # Broadcast: latent_dim → (N, stock_embed_dim)
        self.broadcast = nn.Sequential(
            nn.Linear(latent_dim, n_stocks * 64), nn.GELU(),
        )

        # Per-stock decode (shared weights): 64 → (W·F)
        self.stock_dec = nn.Sequential(
            nn.Linear(64, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, out_dim),
        )

    def forward(self, z):
        # z: (B, latent_dim)
        B = z.shape[0]
        s = self.broadcast(z).reshape(B * self.N, 64)       # (B·N, 64)
        x = self.stock_dec(s)                               # (B·N, W·F)
        return x.reshape(B, self.N, self.W, self.F)         # (B, N, W, F)


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# DEEP KOOPMAN SYSTEM  —  the full model
# ─────────────────────────────────────────────────────────────────────────────
class DeepKoopmanSystem(nn.Module):
    """
    Encapsulates the four core objects of the Koopman system:

      Encoder ϕ  : xₜ (N,W,F) → zₜ (latent_dim,)
      K          : zₜ (latent_dim,) → K·zₜ  [market autonomous dynamics]
      B          : uₜ (N_stocks,) → B·uₜ    [control influence on state]
      C          : zₜ (latent_dim,) → μₜ (N_stocks,)  [return readout]
      Decoder ψ  : zₜ (latent_dim,) → x̂ₜ (N,W,F)

    State equation enforced inside the bottleneck:
        ẑₜ₊₁ = K·zₜ + B·uₜ
    """

    def __init__(self, window, n_features, n_stocks,
                 stock_embed_dim=64, latent_dim=64, dropout=0.1):
        super().__init__()
        self.encoder = KoopmanEncoder(window, n_features, stock_embed_dim, latent_dim, dropout)
        self.decoder = KoopmanDecoder(window, n_features, n_stocks, latent_dim, dropout)

        # ── K: Koopman transition operator ───────────────────────────────────
        # Plain Linear, no spectral_norm — stability enforced via eigenvalue
        # regularisation in the loss (differentiable, does not globally cap K).
        # Initialised ≈ Identity + small noise: warm start where "tomorrow ≈ today".
        self.K = nn.Linear(latent_dim, latent_dim, bias=False)
        nn.init.eye_(self.K.weight)
        with torch.no_grad():
            self.K.weight.add_(torch.randn_like(self.K.weight) * 0.01)

        # ── B: Control input matrix ───────────────────────────────────────────
        # Maps portfolio weights uₜ ∈ ℝ^{N_stocks} → latent perturbation ∈ ℝ^{latent_dim}.
        # Learnt jointly: the encoder signals what changed in the latent state when
        # a particular portfolio allocation was active.
        self.B = nn.Linear(n_stocks, latent_dim, bias=False)
        nn.init.xavier_uniform_(self.B.weight, gain=0.1)   # start small — B is a correction

        # ── C: Return readout matrix ──────────────────────────────────────────
        # Maps latent state zₜ → expected per-stock returns μₜ.
        # Supervised by actual log returns, so C learns the return-predictive
        # directions in latent space. Used by MPC to compute expected portfolio return.
        self.C = nn.Linear(latent_dim, n_stocks, bias=False)
        nn.init.xavier_uniform_(self.C.weight, gain=0.1)

        self.latent_dim = latent_dim
        self.n_stocks   = n_stocks

    def forward(self, x_t, x_t1, u_t):
        """
        Args:
            x_t  : (B, N, W, F)  — current observation window
            x_t1 : (B, N, W, F)  — next observation window
            u_t  : (B, N)         — portfolio weights at time t
        Returns a dict with all quantities needed for the 4-part loss.
        """
        # Encode both windows to latent states
        z_t   = self.encoder(x_t)    # (B, latent_dim)
        z_t1  = self.encoder(x_t1)   # (B, latent_dim)  ← ground truth next latent

        # Apply Koopman state equation:  ẑₜ₊₁ = K·zₜ + B·uₜ
        z_t1_hat = self.K(z_t) + self.B(u_t)   # (B, latent_dim)

        # Decode
        x_t_recon = self.decoder(z_t)       # (B, N, W, F)  — reconstruction
        x_t1_pred = self.decoder(z_t1_hat)  # (B, N, W, F)  — forward prediction

        # Return readout:  μₜ = C·zₜ  (expected returns given current state)
        mu_t = self.C(z_t)                  # (B, N_stocks)

        return dict(z_t=z_t, z_t1=z_t1, z_t1_hat=z_t1_hat,
                    x_t_recon=x_t_recon, x_t1_pred=x_t1_pred,
                    x_t=x_t, x_t1=x_t1, mu_t=mu_t)

    @torch.no_grad()
    def eigval_penalty(self):
        """Soft eigenvalue regularisation: penalise |λ(K)| > 1.
        ReLU(|λ|−1)² → zero inside unit circle, positive outside.
        Replaces hard spectral_norm constraint, preserving gradient flow."""
        eigs = torch.linalg.eigvals(self.K.weight)  # complex tensor
        return F.relu(eigs.abs() - 1.0).pow(2).mean()


In [ ]:

# ── Sanity check ─────────────────────────────────────────────────────────────
model_test = DeepKoopmanSystem(
    window=WINDOW, n_features=N_FEATURES, n_stocks=N_STOCKS,
    stock_embed_dim=STOCK_EMBED_DIM, latent_dim=LATENT_DIM, dropout=DROPOUT
)

x_dummy  = torch.zeros(2, N_STOCKS, WINDOW, N_FEATURES)
u_dummy  = torch.zeros(2, N_STOCKS)
out_test = model_test(x_dummy, x_dummy, u_dummy)

n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Model parameters  : {n_params:,}')
print(f'  Encoder         : {sum(p.numel() for p in model_test.encoder.parameters()):,}')
print(f'  Decoder         : {sum(p.numel() for p in model_test.decoder.parameters()):,}')
print(f'  K ({LATENT_DIM}×{LATENT_DIM})     : {LATENT_DIM**2:,}')
print(f'  B ({LATENT_DIM}×{N_STOCKS})  : {LATENT_DIM*N_STOCKS:,}')
print(f'  C ({N_STOCKS}×{LATENT_DIM})  : {N_STOCKS*LATENT_DIM:,}')
print(f'\nForward pass shapes:')
print(f'  z_t      : {tuple(out_test["z_t"].shape)}')
print(f'  z_t1_hat : {tuple(out_test["z_t1_hat"].shape)}')
print(f'  mu_t     : {tuple(out_test["mu_t"].shape)}')
print(f'  x_t_recon: {tuple(out_test["x_t_recon"].shape)}')
del model_test, out_test, x_dummy, u_dummy

## Phase 1 — Step 5: Loss Function

Four-component loss — each component trains a different matrix:

| Loss | Formula | Trains |
|------|---------|--------|
| **L_recon** | `‖ψ(ϕ(xₜ)) − xₜ‖²` | ϕ, ψ — valid autoencoder |
| **L_latent** | `‖Kzₜ + Buₜ − ϕ(xₜ₊₁)‖²` | K, B — core Koopman constraint |
| **L_forward** | `‖ψ(Kzₜ + Buₜ) − xₜ₊₁‖²` | All — predictions meaningful in obs. space |
| **L_return** | `‖Czₜ − rₜ₊₁‖²` | C — return readout is predictive |

Plus the **eigenvalue regularisation** on K (soft constraint, not a loss component).

In [ ]:
class KoopmanSystemLoss(nn.Module):
    """
    L = λ_r·L_recon + λ_l·L_latent + λ_f·L_forward + λ_ret·L_return + λ_e·L_eigval

    Gradient note: L_latent uses .detach() on z_t1 (the encoded ground truth).
    This means K and B are pushed toward the ENCODED target, not the other way
    around — preventing the encoder from trivially collapsing to K's output.
    """
    def __init__(self, lam_recon=1.0, lam_latent=4.0,
                 lam_fwd=1.5, lam_ret=2.0, lam_eigval=0.5):
        super().__init__()
        self.lr, self.ll, self.lf = lam_recon, lam_latent, lam_fwd
        self.lret, self.le       = lam_ret, lam_eigval

    def forward(self, out: dict, r_t1: torch.Tensor, eigval_reg: torch.Tensor):
        """
        Args:
            out        : dict from model.forward()
            r_t1       : (B, N) actual next-day log returns
            eigval_reg : scalar penalty from model.eigval_penalty()
        """
        z_t, z_t1, z_t1_hat     = out['z_t'], out['z_t1'], out['z_t1_hat']
        x_t_recon, x_t1_pred    = out['x_t_recon'], out['x_t1_pred']
        x_t, x_t1               = out['x_t'], out['x_t1']
        mu_t                     = out['mu_t']

        # 1. Reconstruction: autoencoder fidelity
        L_recon  = F.mse_loss(x_t_recon, x_t)

        # 2. Latent linearisation: CORE Koopman constraint
        #    Kzₜ + Buₜ  ≈  ϕ(xₜ₊₁)   in latent space
        #    .detach() on z_t1 → asymmetric gradient (K/B move, encoder stays put)
        L_latent = F.mse_loss(z_t1_hat, z_t1.detach())

        # 3. Forward prediction: decoded prediction matches next observation
        L_fwd    = F.mse_loss(x_t1_pred, x_t1)

        # 4. Return supervision: C·zₜ ≈ rₜ₊₁
        L_ret    = F.mse_loss(mu_t, r_t1)

        # 5. Eigenvalue stability (already computed per-batch, not per-sample)
        L_eigval = eigval_reg

        L_total = (self.lr  * L_recon  +
                   self.ll  * L_latent +
                   self.lf  * L_fwd    +
                   self.lret* L_ret    +
                   self.le  * L_eigval)

        return L_total, dict(recon=L_recon, latent=L_latent,
                             fwd=L_fwd, ret=L_ret, eigval=L_eigval)


print('Loss function defined.')

## Phase 1 — Step 6: Training Loop

**Optimiser**: AdamW with weight decay — regularises K, B, C implicitly.

**Scheduler**: Cosine annealing — smooth LR decay avoids sharp loss spikes.

**Gradient clipping** (`max_norm = 1.0`): Critical for K stability. Large gradients through K can cause eigenvalue explosion mid-training.

**Early stopping**: Monitors total validation loss with patience.

In [ ]:
def train_koopman_system(model, train_loader, val_loader,
                         n_epochs=80, lr=3e-4, patience=15, device='cpu'):

    model     = model.to(device)
    criterion = KoopmanSystemLoss(
        lam_recon=LAMBDA_RECON, lam_latent=LAMBDA_LATENT,
        lam_fwd=LAMBDA_FWD, lam_ret=LAMBDA_RET, lam_eigval=LAMBDA_EIGVAL
    )
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

    keys    = ['total', 'recon', 'latent', 'fwd', 'ret', 'eigval']
    history = {f'{split}_{k}': [] for split in ('train', 'val') for k in keys}

    best_val  = float('inf')
    best_wts  = deepcopy(model.state_dict())
    patience_ctr = 0

    hdr = (f"{'Ep':>4} | {'Tr-Tot':>7} | {'Tr-Lat':>7} | {'Tr-Ret':>7} | "
           f"{'Va-Tot':>7} | {'Va-Lat':>7} | {'Va-Ret':>7} | {'LR':>7}")
    print(hdr); print('-' * len(hdr))

    def run_epoch(loader, training):
        model.train(training)
        acc = {k: 0.0 for k in keys}
        n   = 0
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            for x_t, x_t1, u_t, r_t1 in loader:
                x_t, x_t1 = x_t.to(device), x_t1.to(device)
                u_t, r_t1 = u_t.to(device), r_t1.to(device)

                out     = model(x_t, x_t1, u_t)
                eig_reg = model.eigval_penalty()
                loss, parts = criterion(out, r_t1, eig_reg)

                if training:
                    optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                bs = x_t.shape[0]
                acc['total'] += loss.item() * bs
                for k, v in parts.items():
                    acc[k] += v.item() * bs
                n += bs

        return {k: v / n for k, v in acc.items()}

    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(train_loader, training=True)
        va = run_epoch(val_loader,   training=False)
        scheduler.step()

        for k in keys:
            history[f'train_{k}'].append(tr[k])
            history[f'val_{k}'].append(va[k])

        if va['total'] < best_val - 1e-6:
            best_val     = va['total']
            best_wts     = deepcopy(model.state_dict())
            patience_ctr = 0
            flag = '✓'
        else:
            patience_ctr += 1
            flag = ''

        if epoch % 5 == 0 or epoch == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(f"{epoch:>4} | {tr['total']:>7.4f} | {tr['latent']:>7.4f} | "
                  f"{tr['ret']:>7.4f} | {va['total']:>7.4f} | {va['latent']:>7.4f} | "
                  f"{va['ret']:>7.4f} | {lr_now:.1e} {flag}")

        if patience_ctr >= patience:
            print(f'\n⏹  Early stop at epoch {epoch}  |  best val = {best_val:.5f}')
            break

    model.load_state_dict(best_wts)
    print(f'\n✅ Training complete  |  best val loss = {best_val:.5f}')
    return model, history


print('Training function defined.')

In [ ]:
# ── Instantiate & Train ───────────────────────────────────────────────────────
model = DeepKoopmanSystem(
    window=WINDOW, n_features=N_FEATURES, n_stocks=N_STOCKS,
    stock_embed_dim=STOCK_EMBED_DIM, latent_dim=LATENT_DIM, dropout=DROPOUT
)

model, history = train_koopman_system(
    model, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LR, patience=PATIENCE, device=device
)

# ── Save checkpoint ───────────────────────────────────────────────────────────
torch.save({
    'model_state_dict' : model.state_dict(),
    'config'           : dict(window=WINDOW, n_features=N_FEATURES, n_stocks=N_STOCKS,
                              stock_embed_dim=STOCK_EMBED_DIM, latent_dim=LATENT_DIM),
    'history'          : history,
}, 'koopman_checkpoint.pt')
print('Checkpoint saved → koopman_checkpoint.pt')

In [ ]:
def plot_loss_curves(history):
    components = [
        ('total',  'Total Loss (weighted)',      0, 0),
        ('latent', 'L_latent  (Koopman core)',   0, 1),
        ('recon',  'L_recon   (Autoencoder)',     0, 2),
        ('fwd',    'L_forward (Obs. prediction)', 1, 0),
        ('ret',    'L_return  (C matrix)',        1, 1),
        ('eigval', 'Eigenvalue Stability',        1, 2),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('Koopman System — Training Loss Decomposition', fontsize=14, fontweight='bold')

    for key, title, r, c in components:
        ax = axes[r, c]
        ax.plot(history[f'train_{key}'], lw=2, color='steelblue', label='Train')
        ax.plot(history[f'val_{key}'],   lw=2, color='darkorange', ls='--', label='Val')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(); ax.grid(True, alpha=0.3)
        ax.set_yscale('log')

    plt.tight_layout()
    plt.savefig('koopman_loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Loss curves saved → koopman_loss_curves.png')

    print('\nDiagnostic Guide:')
    print('  L_latent stays high while others fall → K/B not learning dynamics')
    print('  L_return  stays high                  → C not finding return signal')
    print('  Val diverges from Train               → Overfitting; increase dropout')


In [ ]:

plot_loss_curves(history)

## Phase 1 — Step 7: Matrix Extraction & Diagnostics

The neural network phase is now **over**. We extract K, B, C as plain numpy matrices and discard the neural network for Phase 2.

### Diagnostic Checks
- **K eigenspectrum**: All eigenvalues should be inside or near the unit circle. Modes outside → explosive dynamics.
- **B heatmap**: Which stocks most strongly influence latent dynamics? Large |B[:,i]| means stock i has high control authority.
- **C heatmap**: Which latent dimensions are most return-predictive? Reveals the structure of alpha.

In [ ]:
model.eval()

with torch.no_grad():
    K_np = model.K.weight.cpu().numpy()   # (latent_dim, latent_dim)
    B_np = model.B.weight.cpu().numpy()   # (latent_dim, N_stocks)    — B @ u
    C_np = model.C.weight.cpu().numpy()   # (N_stocks,  latent_dim)   — C @ z

print(f'K shape : {K_np.shape}   (latent_dim × latent_dim)')
print(f'B shape : {B_np.shape}  (latent_dim × N_stocks)')
print(f'C shape : {C_np.shape}  (N_stocks × latent_dim)')

# ── K eigenspectrum ───────────────────────────────────────────────────────────
eigvals = np.linalg.eigvals(K_np)   # complex
mags    = np.abs(eigvals)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Eigenvalue unit-circle plot
ax = axes[0]
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k--', lw=1, alpha=0.5, label='Unit circle')
sc = ax.scatter(eigvals.real, eigvals.imag, c=mags, cmap='RdYlGn_r',
                vmin=0.8, vmax=1.2, s=60, zorder=5)
plt.colorbar(sc, ax=ax, label='|λ|')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('K — Eigenvalue Spectrum', fontweight='bold')
ax.set_xlabel('Real'); ax.set_ylabel('Imaginary'); ax.legend()
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)

# B: mean |B| per stock — control authority
ax = axes[1]
b_authority = np.abs(B_np).mean(axis=0)   # (N_stocks,) mean col magnitude
sort_idx    = np.argsort(b_authority)[::-1]
ax.barh([TICKERS[i].split('.')[0] for i in sort_idx],
        b_authority[sort_idx], color='steelblue')
ax.set_xlabel('Mean |B_col|  (control authority)')
ax.set_title('B matrix — Control Authority per Stock', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# C: mean |C| per stock — return predictability
ax = axes[2]
c_strength = np.abs(C_np).mean(axis=1)   # (N_stocks,) mean row magnitude
sort_idx2  = np.argsort(c_strength)[::-1]
ax.barh([TICKERS[i].split('.')[0] for i in sort_idx2],
        c_strength[sort_idx2], color='seagreen')
ax.set_xlabel('Mean |C_row|  (return signal strength)')
ax.set_title('C matrix — Return Readout Strength per Stock', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('koopman_matrix_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nEigenvalue summary:')
print(f'  Max  |λ| : {mags.max():.4f}  (want ≤ 1.05)')
print(f'  Mean |λ| : {mags.mean():.4f}')
print(f'  Outside unit circle : {(mags > 1.0).sum()}/{len(mags)}')

np.save('K_matrix.npy', K_np)
np.save('B_matrix.npy', B_np)
np.save('C_matrix.npy', C_np)
print('\nK, B, C saved as .npy  —  Phase 1 Complete.')

## Phase 2 — Model Predictive Control (The Agent)

The deep learning phase is **over**. We now use K, B, C as a deterministic linear model and solve a convex optimization problem at each time step.

### Why the problem is convex

At time $t$, we propagate the latent state **autonomously** (open-loop, without B) to get a fixed predicted trajectory:
$$\mathbf{z}_{t+k} = \mathbf{K}^k \mathbf{z}_t \quad \Rightarrow \quad \boldsymbol{\mu}_{t+k} = \mathbf{C}\mathbf{z}_{t+k} \in \mathbb{R}^N$$

With $\boldsymbol{\mu}_{t+k}$ fixed (a constant vector at step $k$), the MPC objective becomes:
$$J = \max_{\{\mathbf{u}_k\}} \sum_{k=0}^{H} \left[ \underbrace{\mathbf{u}_k^\top \boldsymbol{\mu}_{t+k}}_{\text{linear in }\mathbf{u}} - \underbrace{\gamma\, \mathbf{u}_k^\top \boldsymbol{\Sigma}\, \mathbf{u}_k}_{\text{concave quadratic}} - \underbrace{\lambda \|\mathbf{u}_k - \mathbf{u}_{k-1}\|_1}_{\text{concave, L1}} \right]$$

This is the sum of concave functions → maximising a concave function → **convex program**. CVXPY solves it to guaranteed global optimality in milliseconds.

The **B matrix** role: B was trained to capture how your trades shift the market's latent state (system identification). Its effect re-enters at the **next encode step** when the model observes the market after you've acted — not in the horizon propagation itself. This is the standard receding-horizon MPC paradigm.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COVARIANCE ESTIMATION
# Use the full training period to estimate a robust Σ (Ledoit-Wolf shrinkage
# approximated here via a simple identity shrinkage for stability).
# During backtesting we update Σ with a rolling window.
# ─────────────────────────────────────────────────────────────────────────────

def estimate_sigma(log_returns: np.ndarray, shrinkage: float = 0.1) -> np.ndarray:
    """
    Compute shrunk sample covariance matrix.
    Σ_shrunk = (1 - α)·Σ_sample + α·(trace(Σ)/N)·I

    Shrinkage towards scaled identity improves condition number, preventing
    the MPC from concentrating in poorly-estimated low-variance directions.

    Args:
        log_returns : (T, N) matrix of daily log returns
        shrinkage   : α ∈ [0, 1]  (0 = sample only, 1 = identity only)
    Returns:
        Σ : (N, N) positive-definite covariance matrix
    """
    T, N   = log_returns.shape
    mu     = log_returns.mean(axis=0)
    demeaned = log_returns - mu
    Sigma_s  = (demeaned.T @ demeaned) / (T - 1)      # sample covariance
    target   = (np.trace(Sigma_s) / N) * np.eye(N)    # scaled identity target
    return (1 - shrinkage) * Sigma_s + shrinkage * target


In [ ]:

# ── Compute training-period Σ ─────────────────────────────────────────────────
train_log_returns = data_train[:, :, LOGRET_IDX]   # (T_train, N_stocks)
Sigma_train = estimate_sigma(train_log_returns, shrinkage=0.1)

print(f'Σ shape       : {Sigma_train.shape}')
print(f'Σ condition # : {np.linalg.cond(Sigma_train):.1f}  (lower is better; <1000 is fine)')
eigvals_sigma = np.linalg.eigvalsh(Sigma_train)
print(f'Min eigenvalue: {eigvals_sigma.min():.2e}  (must be > 0 for PD)')

np.save('Sigma_train.npy', Sigma_train)

In [ ]:
class KoopmanMPC:
    """
    Model Predictive Controller that uses the extracted Koopman matrices
    (K, C) and a Markowitz covariance Σ to solve a rolling convex program.

    The MPC problem at time t:

        max   Σ_{k=0}^{H} [ u_k^T μ_{t+k}  -  γ u_k^T Σ u_k  -  λ ‖u_k − u_{k-1}‖₁ ]
        s.t.  u_k^T 1 = 1   (fully invested)
              u_k ≥ 0       (long-only)
              u_k ≤ w_max   (diversification cap)

    where μ_{t+k} = C @ K^k @ z_t  (autonomous Koopman propagation).

    The problem is a Quadratic Program (QP) with L1 penalty on turnover,
    reformulated with auxiliary variables for CVXPY.
    """

    def __init__(self, K: np.ndarray, C: np.ndarray, Sigma: np.ndarray,
                 n_stocks: int, H: int = 7,
                 gamma: float = 1.0, lambda_tc: float = 0.001,
                 max_weight: float = 0.20,
                 long_only: bool = True):
        self.K, self.C      = K, C
        self.Sigma          = Sigma
        self.N, self.H      = n_stocks, H
        self.gamma          = gamma
        self.lambda_tc      = lambda_tc
        self.max_weight     = max_weight
        self.long_only      = long_only

        # Pre-compute K^k matrices for k = 0 … H
        self._Kpow = [np.eye(K.shape[0])]
        for _ in range(H):
            self._Kpow.append(self._Kpow[-1] @ K)

    def _predicted_returns(self, z0: np.ndarray) -> list:
        """Returns list of H+1 vectors μ_k = C @ K^k @ z0."""
        return [self.C @ Kk @ z0 for Kk in self._Kpow]   # list of (N,) arrays

    def solve(self, z_t: np.ndarray, u_prev: np.ndarray) -> np.ndarray:
        """
        Solve the MPC problem and return the first action u_t*.

        Args:
            z_t    : (latent_dim,) current latent state
            u_prev : (N,) previous portfolio weights  (for transaction cost)
        Returns:
            u_star : (N,) optimal portfolio for the current period
        """
        mus = self._predicted_returns(z_t)   # H+1 vectors of shape (N,)

        # CVXPY decision variables — one weight vector per horizon step
        U = [cp.Variable(self.N, name=f'u_{k}') for k in range(self.H + 1)]

        # L1 turnover: introduce auxiliary variable  s_k ≥ |u_k − u_{k-1}|
        # (CVXPY handles cp.norm1 natively, no manual linearisation needed)
        objective_terms = []
        for k in range(self.H + 1):
            u_km1 = u_prev if k == 0 else U[k - 1]

            expected_ret  = mus[k] @ U[k]                          # scalar, linear
            markowitz_risk= cp.quad_form(U[k], self.Sigma)         # scalar, convex
            turnover_cost = cp.norm1(U[k] - u_km1)                 # scalar, convex

            # Objective contribution for step k (we MAXIMISE, so risk/cost enter negatively)
            term = expected_ret - self.gamma * markowitz_risk - self.lambda_tc * turnover_cost
            objective_terms.append(term)

        obj = cp.Maximize(sum(objective_terms))

        # Constraints (applied to every u_k in the horizon)
        constraints = []
        for k in range(self.H + 1):
            constraints.append(cp.sum(U[k]) == 1)                  # fully invested
            if self.long_only:
                constraints.append(U[k] >= 0)
            constraints.append(U[k] <= self.max_weight)            # diversification

        prob = cp.Problem(obj, constraints)
        try:
            prob.solve(solver=cp.CLARABEL, verbose=False)
        except Exception:
            # Fallback to ECOS if CLARABEL unavailable
            prob.solve(solver=cp.ECOS, verbose=False)

        # If solver fails, fall back to equal-weight portfolio
        if prob.status not in ('optimal', 'optimal_inaccurate') or U[0].value is None:
            return np.full(self.N, 1.0 / self.N)

        # Return only the FIRST action (receding horizon principle)
        u_star = np.array(U[0].value, dtype=np.float64)
        # Clip small negatives from solver tolerance, renormalise
        u_star = np.clip(u_star, 0, 1)
        u_star /= u_star.sum()
        return u_star


In [ ]:

# ── Quick solve test ──────────────────────────────────────────────────────────
print('Testing MPC solver...')
mpc_test = KoopmanMPC(
    K=K_np, C=C_np, Sigma=Sigma_train, n_stocks=N_STOCKS,
    H=H, gamma=GAMMA, lambda_tc=LAMBDA_TC, max_weight=MAX_WEIGHT
)
z_dummy = np.random.randn(LATENT_DIM)
u_dummy = np.full(N_STOCKS, 1.0 / N_STOCKS)
u_test  = mpc_test.solve(z_dummy, u_dummy)

print(f'Solver status : OK')
print(f'u_star sum    : {u_test.sum():.6f}  (should be 1.0)')
print(f'u_star min    : {u_test.min():.4f}  (should be ≥ 0)')
print(f'u_star max    : {u_test.max():.4f}  (should be ≤ {MAX_WEIGHT})')
print(f'Non-zero positions : {(u_test > 1e-4).sum()}')
del z_dummy, u_dummy, u_test, mpc_test

## Phase 3 — Convex Execution: Rolling Backtest

### Per-Day Loop
1. Observe market window `x_t` → **Encode** → latent state `z_t`
2. Pass `z_t`, K, C, Σ into **CVXPY** MPC solver
3. CVXPY returns mathematically guaranteed global-optimal `u_t*` in milliseconds
4. **Execute only `u_t*`** in the real market
5. Roll forward to t+1, repeat

**Transaction costs**: 10 bps per unit of turnover (|Δweight|) applied to all backtest returns.

**Covariance**: Rolling 60-day window, updated each period.

In [ ]:
def encode_window(model, window_np: np.ndarray, device) -> np.ndarray:
    """Encode a single (N, W, F) numpy window → z (latent_dim,) numpy."""
    model.eval()
    with torch.no_grad():
        x = torch.tensor(window_np, dtype=torch.float32).unsqueeze(0).to(device)  # (1, N, W, F)
        z = model.encoder(x)   # (1, latent_dim)
    return z.squeeze(0).cpu().numpy()


In [ ]:

def run_backtest(model, data_norm, log_returns_raw, K, C, Sigma_global,
                 tickers, window, cov_window, H, gamma, lambda_tc, max_weight, device):
    """
    Full rolling MPC backtest.

    Returns a DataFrame with columns:
      ret_mpc_gross, ret_mpc_net, cost, ret_ew (equal-weight benchmark)
    """
    N  = len(tickers)
    T  = data_norm.shape[0]

    results   = []
    u_prev    = np.full(N, 1.0 / N)   # start from equal weight
    mpc       = KoopmanMPC(K=K, C=C, Sigma=Sigma_global, n_stocks=N,
                           H=H, gamma=gamma, lambda_tc=lambda_tc, max_weight=max_weight)

    # First valid signal: we need a full window of W days
    # Day t: window = data_norm[t-W : t], signal realized on day t
    print(f'Running MPC backtest over {T - window} signal days...')

    for t in range(window, T):
        # ── 1. Observe & Encode ───────────────────────────────────────────────
        window_np = data_norm[t - window : t]   # (W, N, F)
        window_np = window_np.transpose(1, 0, 2)  # (N, W, F)
        z_t = encode_window(model, window_np, device)   # (latent_dim,)

        # ── 2. Rolling covariance update ──────────────────────────────────────
        cov_start = max(0, t - cov_window)
        recent_ret = log_returns_raw[cov_start:t, :]   # (≤cov_window, N)
        if recent_ret.shape[0] >= 10:   # need at least 10 observations
            Sigma_t = estimate_sigma(recent_ret, shrinkage=0.1)
            mpc.Sigma = Sigma_t
        # else: keep global Sigma

        # ── 3. Solve MPC → optimal weights ───────────────────────────────────
        u_star = mpc.solve(z_t, u_prev)

        # ── 4. Compute realised return on day t ───────────────────────────────
        r_t = log_returns_raw[t, :]   # (N,) actual log returns on day t

        # Portfolio return (log-space, pre-cost)
        ret_gross = float(u_star @ r_t)

        # Equal-weight benchmark return
        ret_ew = float(r_t.mean())

        # Transaction cost: 10 bps per unit of absolute weight change
        turnover = np.abs(u_star - u_prev).sum()
        cost     = turnover * LAMBDA_TC
        ret_net  = ret_gross - cost

        results.append({
            'ret_mpc_gross': ret_gross,
            'ret_mpc_net'  : ret_net,
            'cost'         : cost,
            'ret_ew'       : ret_ew,
            'turnover'     : turnover,
        })

        # ── 5. Weight memory for next step ────────────────────────────────────
        u_prev = u_star

        if (t - window + 1) % 100 == 0:
            print(f'  Day {t - window + 1}/{T - window}  |  last u_star max={u_star.max():.3f}')

    print('Backtest complete.')
    return pd.DataFrame(results)


print('Backtest function defined.')

In [ ]:
# ── Prepare test-period raw log returns (denormalised) ────────────────────────
test_log_returns_raw = (
    data_test_n[:, :, LOGRET_IDX] * train_std[LOGRET_IDX] + train_mean[LOGRET_IDX]
)  # (T_test, N_stocks)  — actual daily log returns in test period

date_index_test = pd.to_datetime(
    np.load('date_index_test.npy', allow_pickle=True)
)

# ── Run backtest ──────────────────────────────────────────────────────────────
bt_df = run_backtest(
    model=model,
    data_norm=data_test_n,
    log_returns_raw=test_log_returns_raw,
    K=K_np, C=C_np, Sigma_global=Sigma_train,
    tickers=TICKERS,
    window=WINDOW,
    cov_window=COV_WINDOW,
    H=H,
    gamma=GAMMA,
    lambda_tc=LAMBDA_TC,
    max_weight=MAX_WEIGHT,
    device=device,
)

# Attach dates (signal dates start at WINDOW-th day of test period)
signal_dates = date_index_test[WINDOW:]
bt_df.index  = signal_dates[:len(bt_df)]
bt_df.index.name = 'date'

# ── Add BSE Sensex benchmark ──────────────────────────────────────────────────
bse = yf.download(SENSEX_BENCHMARK, start=date_index_test[0], end=date_index_test[-1],
                  auto_adjust=True, progress=False)
bse_ret = np.log(bse['Close'] / bse['Close'].shift(1)).dropna()
bt_df['ret_bench'] = bse_ret.reindex(bt_df.index).fillna(0).values

bt_df.to_csv('mpc_backtest_results.csv')
print(f'\nBacktest shape : {bt_df.shape}')
print(bt_df.head())

## Phase 3 — Performance Analytics

In [ ]:
def performance_stats(daily_log_returns: pd.Series, ann_factor: int = 252) -> dict:
    """Compute annualised performance statistics from a daily log-return series."""
    r  = daily_log_returns.dropna()
    cum_wealth = np.exp(r.cumsum())

    # CAGR
    total_ret  = cum_wealth.iloc[-1] - 1
    n_years    = len(r) / ann_factor
    cagr       = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0

    # Volatility & Sharpe  (annualised, log-return approximation)
    ann_vol    = r.std() * np.sqrt(ann_factor)
    ann_ret    = r.mean() * ann_factor
    sharpe     = ann_ret / (ann_vol + 1e-9)

    # Maximum drawdown
    running_max = cum_wealth.cummax()
    drawdown    = (cum_wealth - running_max) / running_max
    max_dd      = drawdown.min()

    # Calmar
    calmar     = cagr / (abs(max_dd) + 1e-9)

    # Win rate (% of days with positive return)
    win_rate   = (r > 0).mean()

    return dict(
        CAGR=cagr, Ann_Vol=ann_vol, Sharpe=sharpe,
        Max_DD=max_dd, Calmar=calmar, Win_Rate=win_rate,
        Total_Return=total_ret, N_Days=len(r)
    )


In [ ]:

# ── Compute stats for all strategies ─────────────────────────────────────────
strategies = {
    'MPC (gross)' : bt_df['ret_mpc_gross'],
    'MPC (net)  ' : bt_df['ret_mpc_net'],
    'Equal Weight': bt_df['ret_ew'],
    'BSE Sensex ' : bt_df['ret_bench'],
}

print('=' * 80)
print(f"{'Strategy':<18} {'CAGR':>8} {'Vol':>8} {'Sharpe':>8} "
      f"{'MaxDD':>9} {'Calmar':>8} {'Win%':>7}")
print('-' * 80)
for name, ret_series in strategies.items():
    s = performance_stats(ret_series)
    print(f"{name:<18} {s['CAGR']:>8.2%} {s['Ann_Vol']:>8.2%} "
          f"{s['Sharpe']:>8.2f} {s['Max_DD']:>9.2%} "
          f"{s['Calmar']:>8.2f} {s['Win_Rate']:>7.1%}")
print('=' * 80)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
fig.suptitle('Koopman MPC — Test Period Performance (2023–2024)', fontsize=14, fontweight='bold')

# ── Panel 1: Cumulative wealth ─────────────────────────────────────────────
ax = axes[0]
cum = {name: np.exp(ret.cumsum()) for name, ret in strategies.items()}
colors  = ['steelblue', 'royalblue', 'seagreen', 'darkorange']
styles  = ['-', '--', '-.', ':']
for (name, c), col, ls in zip(cum.items(), colors, styles):
    ax.plot(c.index, c.values, lw=2, color=col, ls=ls, label=name.strip())
ax.axhline(1.0, color='grey', lw=0.8, ls=':')
ax.set_ylabel('Wealth Index (start = 1.0)')
ax.set_title('Cumulative Wealth')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)

# ── Panel 2: Drawdown ─────────────────────────────────────────────────────
ax = axes[1]
for (name, c), col in zip(cum.items(), colors):
    running_max = c.cummax()
    dd = (c - running_max) / running_max
    ax.fill_between(dd.index, dd.values, 0, alpha=0.35, color=col, label=name.strip())
ax.set_ylabel('Drawdown')
ax.set_title('Drawdown Profile')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)

# ── Panel 3: Rolling 30-day Sharpe of MPC net vs benchmark ───────────────
ax = axes[2]
roll = bt_df[['ret_mpc_net', 'ret_bench']].rolling(30)
sharpe_mpc   = (roll['ret_mpc_net'].mean()   / (roll['ret_mpc_net'].std()   + 1e-9)) * np.sqrt(252)
sharpe_bench = (roll['ret_bench'].mean() / (roll['ret_bench'].std() + 1e-9)) * np.sqrt(252)
ax.plot(sharpe_mpc.index,   sharpe_mpc.values,   lw=2, color='royalblue', label='MPC (net)')
ax.plot(sharpe_bench.index, sharpe_bench.values,  lw=2, color='darkorange', ls='--', label='BSE Sensex')
ax.axhline(0, color='grey', lw=0.8)
ax.set_ylabel('Rolling 30-day Sharpe (annualised)')
ax.set_title('Rolling Sharpe Ratio')
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig('mpc_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → mpc_equity_curves.png')

In [ ]:
# ── Information Coefficient (IC) — signal quality ─────────────────────────
# IC = Spearman rank correlation between C·z_t (predicted returns) and actual returns.
# IC > 0.05 is considered usable; ICIR > 0.3 is good.

print('Computing IC (Information Coefficient) ...')
ic_series = []

model.eval()
with torch.no_grad():
    for t in range(WINDOW, data_test_n.shape[0]):
        win_np  = data_test_n[t - WINDOW : t].transpose(1, 0, 2)   # (N, W, F)
        z_t     = encode_window(model, win_np, device)               # (latent_dim,)
        mu_pred = C_np @ z_t                                         # (N,) predicted returns
        mu_act  = test_log_returns_raw[t, :]                         # (N,) actual returns

        if mu_pred.std() > 1e-8 and mu_act.std() > 1e-8:
            ic, _ = spearmanr(mu_pred, mu_act)
            ic_series.append(ic)
        else:
            ic_series.append(np.nan)

ic_arr  = np.array(ic_series)
ic_mean = np.nanmean(ic_arr)
ic_std  = np.nanstd(ic_arr)
icir    = ic_mean / (ic_std + 1e-9)

print(f'\n{"="*50}')
print(f'  SIGNAL QUALITY — C matrix (Return Readout)')
print(f'{"="*50}')
print(f'  Mean IC   : {ic_mean:+.4f}   (target > 0.05)')
print(f'  IC Std    : {ic_std:.4f}')
print(f'  ICIR      : {icir:+.4f}   (target > 0.30)')
print(f'  IC > 0    : {np.nanmean(ic_arr > 0)*100:.1f}%   (target > 50%)')
print(f'{"="*50}')

# ── IC plot ────────────────────────────────────────────────────────────────
ic_dates = date_index_test[WINDOW : WINDOW + len(ic_arr)]
ic_s     = pd.Series(ic_arr, index=ic_dates)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Signal Quality Diagnostics', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(ic_dates, ic_arr, color=np.where(ic_arr > 0, 'steelblue', 'tomato'), alpha=0.6, width=1)
ax.plot(ic_s.rolling(20).mean(), color='black', lw=2, label='20-day rolling IC')
ax.axhline(0, color='grey', lw=0.8)
ax.axhline(0.05, color='green', lw=1.5, ls='--', label='IC = 0.05 target')
ax.set_title('Daily IC (C·zₜ vs Actual Returns — Spearman ρ)')
ax.set_ylabel('IC'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(bt_df.index, bt_df['turnover'].rolling(20).mean(), lw=2, color='royalblue')
ax.set_title('Mean Portfolio Turnover (20-day rolling)')
ax.set_ylabel('Turnover (Σ|Δu|)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('signal_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → signal_quality.png')

In [ ]:
# ── Weight attribution & allocation analysis ──────────────────────────────
# Reconstruct the weight matrix by re-running the encoder on the test window.
# (For a full-fidelity run, store weights during backtest — this is a rerun
#  on the first 200 test days for visualisation purposes.)

PLOT_DAYS = min(200, len(bt_df))
weight_matrix = np.zeros((PLOT_DAYS, N_STOCKS))
u_prev        = np.full(N_STOCKS, 1.0 / N_STOCKS)

mpc_vis = KoopmanMPC(
    K=K_np, C=C_np, Sigma=Sigma_train, n_stocks=N_STOCKS,
    H=H, gamma=GAMMA, lambda_tc=LAMBDA_TC, max_weight=MAX_WEIGHT
)

model.eval()
with torch.no_grad():
    for i in range(PLOT_DAYS):
        t      = WINDOW + i
        win_np = data_test_n[t - WINDOW : t].transpose(1, 0, 2)
        z_t    = encode_window(model, win_np, device)
        u_star = mpc_vis.solve(z_t, u_prev)
        weight_matrix[i] = u_star
        u_prev = u_star

# Stacked area chart of top-N holdings
weight_df = pd.DataFrame(
    weight_matrix,
    columns=[t.split('.')[0] for t in TICKERS],
    index=bt_df.index[:PLOT_DAYS]
)

# Sort by mean allocation
mean_alloc = weight_df.mean().sort_values(ascending=False)
top_n = min(10, N_STOCKS)
top_tickers = mean_alloc.index[:top_n].tolist()
other = weight_df.drop(columns=top_tickers).sum(axis=1)
plot_df = weight_df[top_tickers].copy()
plot_df['Others'] = other

fig, ax = plt.subplots(figsize=(14, 6))
plot_df.plot.area(ax=ax, alpha=0.8, colormap='tab20')
ax.set_title(f'MPC Portfolio Allocation — First {PLOT_DAYS} Days (Test Period)',
             fontweight='bold')
ax.set_ylabel('Portfolio Weight')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(loc='upper right', ncol=4, fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('weight_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop-5 mean allocations:')
for tk, w in mean_alloc.head(5).items():
    print(f'  {tk:<18} {w:.2%}')
print(f'\nPlot saved → weight_allocation.png')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────
print('\n' + '═' * 60)
print('  DEEP KOOPMAN MPC — FINAL RESULTS SUMMARY')
print('═' * 60)

for name, ret_series in strategies.items():
    s = performance_stats(ret_series)
    print(f'\n  {name.strip()}')
    print(f'    CAGR          : {s["CAGR"]:>8.2%}')
    print(f'    Annualised Vol: {s["Ann_Vol"]:>8.2%}')
    print(f'    Sharpe Ratio  : {s["Sharpe"]:>8.2f}')
    print(f'    Max Drawdown  : {s["Max_DD"]:>8.2%}')
    print(f'    Calmar Ratio  : {s["Calmar"]:>8.2f}')
    print(f'    Win Rate      : {s["Win_Rate"]:>8.1%}')

print(f'\n  IC (C matrix signal quality):')
print(f'    Mean IC : {ic_mean:+.4f}   ICIR : {icir:+.4f}')
print(f'    IC > 0  : {np.nanmean(ic_arr > 0)*100:.1f}%')

print(f'\n  Mean daily turnover (MPC)   : {bt_df["turnover"].mean():.3f}')
print(f'  Mean daily cost  (MPC)      : {bt_df["cost"].mean()*100:.4f}%')
print('═' * 60)
print('\nAll artefacts saved:')
for f in ['koopman_checkpoint.pt', 'K_matrix.npy', 'B_matrix.npy', 'C_matrix.npy',
          'Sigma_train.npy', 'mpc_backtest_results.csv',
          'koopman_loss_curves.png', 'koopman_matrix_diagnostics.png',
          'mpc_equity_curves.png', 'signal_quality.png', 'weight_allocation.png']:
    print(f'  ✓ {f}')